In [4]:
import re
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from transformers import pipeline
import numpy as np

# --------------------------------
# 0. Configurations
# --------------------------------
INPUT_FILE = 'Data1-clean.csv'
CLEANED_FILE = 'cleaning.csv'
SENTIMENT_FILE = 'cleaning_with_sentiment_categorized.csv'
OUTPUT_FILE = 'final_sentiment_categorized.csv'

# --------------------------------
# 1. Improved Text Cleaning & Normalization
# --------------------------------

# Load data
df = pd.read_csv(INPUT_FILE)

# Enhanced normalization dictionary
normalization_dict = {
    # --- Singkatan Umum ---
    'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk', 'kpd': 'kepada',
    'dr': 'dari', 'karna': 'karena', 'krn': 'karena', 'bkn': 'bukan',
    'jg': 'juga', 'sdh': 'sudah', 'udh': 'sudah', 'udah': 'sudah',
    'blm': 'belum', 'thn': 'tahun', 'tgl': 'tanggal', 'dll': 'dan lain lain',
    'dsb': 'dan sebagainya', 'sbg': 'sebagai', 'tdk': 'tidak',
    'ga': 'tidak', 'gak': 'tidak', 'nggak': 'tidak', 'jgn': 'jangan',
    'bnyk': 'banyak', 'dpt': 'dapat', 'aja': 'saja', 'sja': 'saja',
    'trus': 'terus', 'trs': 'terus',
    
    # --- Kata Ganti Orang ---
    'gw': 'saya', 'gue': 'saya', 'ane': 'saya', 'sy': 'saya',
    'lu': 'kamu', 'loe': 'kamu', 'klen': 'kalian',

    # --- Bahasa Gaul & Ekspresi Sentimen Positif ---
    'bgt': 'banget', 'mantul': 'mantap', 'keren': 'bagus', 'oke': 'baik',
    'bagus': 'baik', 'mantap': 'baik', 'top': 'baik', 'recommended': 'bagus',
    'gokil': 'luar biasa', 'amazing': 'luar biasa', 'wow': 'luar biasa',
    'perfect': 'sempurna', 'excellent': 'sangat baik',
    
    # --- Ekspresi Sentimen Negatif ---
    'jelek': 'buruk', 'parah': 'buruk', 'gaje': 'buruk', 'zonk': 'buruk',
    'disappointing': 'mengecewakan', 'terrible': 'buruk', 'awful': 'buruk',
    'hate': 'benci', 'sucks': 'buruk', 'worst': 'terburuk',
    'rubbish': 'sampah', 'useless': 'tidak berguna',

    # --- Koreksi Ejaan & Typo ---
    'cuman': 'cuma', 'emang': 'memang', 'kalo': 'kalau', 'klo': 'kalau',
    'pake': 'pakai', 'jd': 'jadi', 'lg': 'lagi', 'emg': 'memang',
    'kyk': 'seperti', 'kek': 'seperti', 'hny': 'hanya',
    
    # --- Kata Tanya & Sambung ---
    'knp': 'kenapa', 'gmn': 'bagaimana', 'kpn': 'kapan', 'brp': 'berapa',
    'sblm': 'sebelum', 'stlh': 'setelah',
}

# Enhanced stopword list - removing sentiment-bearing words
stop_factory = StopWordRemoverFactory()
default_stopwords = set(stop_factory.get_stop_words())

# Remove sentiment-bearing words from stopwords
sentiment_words = {
    'tidak', 'bukan', 'jangan', 'kurang', 'buruk', 'jelek', 'bagus', 'baik', 
    'suka', 'senang', 'sedih', 'marah', 'kecewa', 'puas', 'mantap', 'hebat',
    'luar', 'biasa', 'sempurna', 'terburuk', 'terbaik', 'recommended'
}
custom_stopwords = default_stopwords - sentiment_words

# Initialize stemmer
stem_factory = StemmerFactory()
stemmer = stem_factory.create_stemmer()

def clean_text_improved(text):
    """Improved text cleaning function that preserves sentiment"""
    if pd.isna(text):
        return ''
    
    # Convert to lowercase
    text = str(text).lower()
    
    # Remove URLs but keep the sentiment context
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Handle mentions and hashtags differently - they might contain sentiment
    text = re.sub(r'@\w+', '', text)  # Remove mentions
    text = re.sub(r'#(\w+)', r'\1', text)  # Keep hashtag content, remove #
    
    # Handle repeated characters (e.g., "bagusssss" -> "bagus banget")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove numbers but keep text structure
    text = re.sub(r'\d+', '', text)
    
    # Handle punctuation more carefully
    text = re.sub(r'[!]{2,}', ' sangat ', text)  # Multiple ! indicates emphasis
    text = re.sub(r'[?]{2,}', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenization and normalization
    tokens = text.split()
    
    # Apply normalization dictionary
    tokens = [normalization_dict.get(tok, tok) for tok in tokens]
    
    # Remove very short tokens (< 2 chars) but keep negation words
    tokens = [tok for tok in tokens if len(tok) >= 2 or tok in ['ga', 'ng']]
    
    # Custom stopword removal (preserving sentiment words)
    tokens = [tok for tok in tokens if tok not in custom_stopwords]
    
    # Light stemming - avoid over-stemming sentiment words
    sentiment_preserve = {'bagus', 'buruk', 'jelek', 'baik', 'suka', 'benci', 'senang', 'sedih'}
    tokens = [tok if tok in sentiment_preserve else stemmer.stem(tok) for tok in tokens]
    
    return ' '.join(tokens)

# Apply improved cleaning
print("Applying text cleaning...")
df['text_cleaned'] = df['cleaned_text'].apply(clean_text_improved)

# Filter out very short texts (likely uninformative)
df = df[df['text_cleaned'].str.len() > 10].reset_index(drop=True)

# Save cleaned text
df.to_csv(CLEANED_FILE, index=False)
print(f"Cleaned data saved to {CLEANED_FILE}")

# --------------------------------
# 2. Improved Sentiment Analysis
# --------------------------------

# Try multiple Indonesian sentiment models for better accuracy
try:
    # First choice: More recent Indonesian sentiment model
    sentiment_analyzer = pipeline(
        "sentiment-analysis",
        model="ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa",
        device=-1,
        return_all_scores=True
    )
    model_type = "ayameRushia"
    print("Using ayameRushia model")
except:
    try:
        # Second choice: IndoBERT sentiment
        sentiment_analyzer = pipeline(
            "sentiment-analysis", 
            model="indobenchmark/indobert-base-p1",
            device=-1,
            return_all_scores=True
        )
        model_type = "indobert"
        print("Using IndoBERT model")
    except:
        # Fallback: Multilingual model
        sentiment_analyzer = pipeline(
            "sentiment-analysis",
            model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
            device=-1,
            return_all_scores=True
        )
        model_type = "xlm-roberta"
        print("Using XLM-RoBERTa model")

def analyze_sentiment_improved(texts, batch_size=16):
    """Improved sentiment analysis with confidence handling"""
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        batch_results = []
        
        for text in batch:
            if len(text.strip()) == 0:
                batch_results.append({
                    'label': 'NEUTRAL',
                    'score': 0.5,
                    'all_scores': [{'label': 'NEUTRAL', 'score': 0.5}]
                })
                continue
                
            try:
                scores = sentiment_analyzer(text)
                if isinstance(scores[0], list):
                    scores = scores[0]
                
                # Find the highest confidence prediction
                best_pred = max(scores, key=lambda x: x['score'])
                
                batch_results.append({
                    'label': best_pred['label'],
                    'score': best_pred['score'],
                    'all_scores': scores
                })
                
            except Exception as e:
                print(f"Error processing text: {e}")
                batch_results.append({
                    'label': 'NEUTRAL',
                    'score': 0.5,
                    'all_scores': [{'label': 'NEUTRAL', 'score': 0.5}]
                })
        
        results.extend(batch_results)
        
        if (i // batch_size + 1) % 10 == 0:
            print(f"Processed {i + len(batch)}/{len(texts)} texts")
    
    return results

# Run sentiment analysis
print("Running sentiment analysis...")
sentiment_results = analyze_sentiment_improved(df['text_cleaned'].tolist())

# Extract results
df['raw_sentiment_label'] = [r['label'] for r in sentiment_results]
df['raw_sentiment_score'] = [r['score'] for r in sentiment_results]

# --------------------------------
# 3. Improved Sentiment Mapping
# --------------------------------

def map_to_final_sentiment(row, confidence_threshold=0.6):
    """
    Map model predictions to positive/negative/neutral with improved logic
    """
    label = row['raw_sentiment_label'].upper()
    score = row['raw_sentiment_score']
    
    # If confidence is too low, classify as neutral
    if score < confidence_threshold:
        return 'neutral'
    
    # Model-specific mapping
    if model_type == "ayameRushia":
        if 'POSITIVE' in label:
            return 'positive'
        elif 'NEGATIVE' in label:
            return 'negative'
        else:
            return 'neutral'
    
    elif model_type == "indobert":
        # IndoBERT typically uses LABEL_0, LABEL_1, etc.
        if label in ['LABEL_2', 'LABEL_3', 'LABEL_4']:  # Higher labels often positive
            return 'positive'
        elif label in ['LABEL_0', 'LABEL_1']:  # Lower labels often negative
            return 'negative'
        else:
            return 'neutral'
    
    elif model_type == "xlm-roberta":
        if 'POSITIVE' in label:
            return 'positive'
        elif 'NEGATIVE' in label:
            return 'negative'
        else:
            return 'neutral'
    
    return 'neutral'

# Apply improved mapping
df['sentiment_category'] = df.apply(map_to_final_sentiment, axis=1)

# --------------------------------
# 4. Post-processing and Validation
# --------------------------------

# Add confidence categories
def categorize_confidence(score):
    if score >= 0.8:
        return 'high'
    elif score >= 0.6:
        return 'medium'
    else:
        return 'low'

df['confidence_level'] = df['raw_sentiment_score'].apply(categorize_confidence)

# Print distribution
print("\n=== SENTIMENT ANALYSIS RESULTS ===")
print("Sentiment Distribution:")
print(df['sentiment_category'].value_counts())
print("\nConfidence Distribution:")
print(df['confidence_level'].value_counts())
print("\nCross-tabulation:")
print(pd.crosstab(df['sentiment_category'], df['confidence_level']))

# Save final results
df.to_csv(OUTPUT_FILE, index=False)
print(f"\nPipeline completed. Final results saved to {OUTPUT_FILE}")

# Show sample results
print("\n=== SAMPLE RESULTS ===")
sample_df = df[['text_cleaned', 'sentiment_category', 'raw_sentiment_score', 'confidence_level']].head(10)
for idx, row in sample_df.iterrows():
    print(f"Text: {row['text_cleaned'][:50]}...")
    print(f"Sentiment: {row['sentiment_category']} (Score: {row['raw_sentiment_score']:.3f}, Confidence: {row['confidence_level']})")
    print("-" * 50)

Applying text cleaning...
Cleaned data saved to cleaning.csv


Device set to use cpu
c:\Users\yusri\Downloads\Penelitian MGB1\.venv\Lib\site-packages\transformers\pipelines\text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Using ayameRushia model
Running sentiment analysis...
Processed 160/6265 texts
Processed 320/6265 texts
Processed 480/6265 texts
Processed 640/6265 texts
Processed 800/6265 texts
Processed 960/6265 texts
Processed 1120/6265 texts
Processed 1280/6265 texts
Processed 1440/6265 texts
Processed 1600/6265 texts
Processed 1760/6265 texts
Processed 1920/6265 texts
Processed 2080/6265 texts
Processed 2240/6265 texts
Processed 2400/6265 texts
Processed 2560/6265 texts
Processed 2720/6265 texts
Processed 2880/6265 texts
Processed 3040/6265 texts
Processed 3200/6265 texts
Processed 3360/6265 texts
Processed 3520/6265 texts
Processed 3680/6265 texts
Processed 3840/6265 texts
Processed 4000/6265 texts
Processed 4160/6265 texts
Processed 4320/6265 texts
Processed 4480/6265 texts
Processed 4640/6265 texts
Processed 4800/6265 texts
Processed 4960/6265 texts
Processed 5120/6265 texts
Processed 5280/6265 texts
Processed 5440/6265 texts
Processed 5600/6265 texts
Processed 5760/6265 texts
Processed 5920/6